começando com a pnad, ideia e ler as duas bases e depois juntar elas do algum jeito q elas se complementem (aquele isco08 que ambas usam ate onde eu li)


In [ ]:
import pandas as pd
import numpy as np
import zipfile
from pathlib import Path
import re
from io import TextIOWrapper


In [ ]:

from pathlib import Path

raiz = Path(
    r"C:\Users\julio\PBL FGV\2ano\4 semestre\econometria\trabalho-econometria1"
)

pasta_dados = raiz / "bases de dados"
pasta_pnad = pasta_dados / "pnad continua"
pasta_oit = pasta_dados / "OIT-IMPACTO DA IA"
pasta_rais = pasta_dados / "RAIS"



for arquivo in pasta_pnad.iterdir():
    if arquivo.is_file():
        print(arquivo.name)



for arquivo in pasta_oit.rglob("*.xlsx"):
    if arquivo.is_file():
        print(arquivo.name)


In [ ]:
variaveis =  [
    # Renda e trabalho
    "VD4016", "V4010", "V4039",
    "VD4002", "VD4009", "V4013",

    # Escolaridade
    "VD3004", "VD3005",

    # Características individuais
    "V2007", "V2009", "V2010",

    # Localização
    "UF", "V1022",

    # Desenho amostral
    "UPA", "Estrato", "V1028"
]


arquivo_layout = next(
    pasta_pnad.glob("*input*.zip"))

with zipfile.ZipFile(arquivo_layout) as z:
    nome_layout = next(
        n for n in z.namelist()
        if "input_pnadc_trimestral" in n.lower()
        and n.lower().endswith(".txt")
    )
    layout = z.read(nome_layout).decode("latin-1")




padrao = r"(?m)^\s*@\s*(\d+)\s+([A-Za-z]\w*)\s+\$?(\d+)\."

campos = re.findall(padrao, layout)

posicoes = {
    nome.upper(): (int(inicio)-1, int(inicio)-1+int(largura))
    for inicio, nome, largura in campos
}







In [ ]:
arquivo_pnad = pasta_pnad / "PNADC_042025.zip"
variaveis_ordenadas = sorted(
    variaveis,
    key=lambda v: posicoes[v.upper()][0]
)
colspecs = [
    posicoes[v.upper()]
    for v in variaveis_ordenadas
]

with zipfile.ZipFile(arquivo_pnad) as z:

    nome_txt = next(
        nome for nome in z.namelist()
        if nome.lower().endswith(".txt")
        and Path(nome).name.upper().startswith("PNADC_")
    )

    with TextIOWrapper(
        z.open(nome_txt),
        encoding="latin-1"
    ) as arquivo:

        pnad = pd.read_fwf(
            arquivo,
            colspecs=colspecs,
            names=variaveis_ordenadas,
            dtype="string",
            nrows=5000
        )

print("Dimensão da base:", pnad.shape)

display(pnad.head()) #!!!!!!

In [ ]:
with zipfile.ZipFile(arquivo_pnad) as z:  #mesmo, mas pra ver o numero total

    with TextIOWrapper(
        z.open(nome_txt),
        encoding="latin-1"
    ) as arquivo:

        pnad = pd.read_fwf(
            arquivo,
            colspecs=colspecs,
            names=variaveis_ordenadas,
            dtype="string"
        )

print("Número de observações:", pnad.shape[0])
print("Número de variáveis:", pnad.shape[1])

display(pnad.head())

In [ ]:
pnad.info()

# Verificar valores ausentes
ausentes = pd.DataFrame({
    "Variável": pnad.columns,
    "Ausentes": pnad.isna().sum().values,
    "% Ausentes": (
        pnad.isna().mean().values * 100
    ).round(2)
})

display(ausentes)

agora o mesmo mas para OIT

In [ ]:
arquivo_oit = next(
    pasta_oit.rglob("Final_Scores_ISCO08_Gmyrek_et_al_2025.xlsx")
)

oit = pd.read_excel(arquivo_oit)
print("Dimensões:", oit.shape)

print("\nColunas disponíveis:")
print(oit.columns.tolist())

display(oit.head())

In [ ]:
colunas_oit = [
    "ISCO_08",
    "Title",
    "mean_score_2025",
    "potential25"
]

oit_ocup = oit[colunas_oit].copy()

verificacao = oit_ocup.groupby("ISCO_08")[
    ["mean_score_2025", "potential25"]
].nunique(dropna=False)

inconsistencias = verificacao[
    (verificacao > 1).any(axis=1)
]

print("Ocupações com valores inconsistentes:", len(inconsistencias))

# Só remover repetições se os indicadores forem consistentes
if not inconsistencias.empty:
    raise ValueError("Existem ocupações com escores divergentes.")

oit_ocup = oit_ocup.drop_duplicates(subset="ISCO_08").copy()

print("Dimensões da nova base:", oit_ocup.shape)

display(oit_ocup.head(10))


In [ ]:
# Verificar as categorias de exposição da OIT
print(oit_ocup["potential25"].value_counts(dropna=False))

# Estatísticas descritivas do escore contínuo
display(oit_ocup["mean_score_2025"].describe())


In [ ]:
oit_ocup["score_ia"] = pd.to_numeric(
    oit_ocup["mean_score_2025"],
    errors="coerce"
)
alta = [
    "Exposed: Gradient 3",
    "Exposed: Gradient 4"
]
baixa = [
    "Not Exposed",
    "Minimal Exposure"
]
oit_ocup["dummy_ia"] = pd.Series(
    pd.NA,
    index=oit_ocup.index,
    dtype="Int64"
)

oit_ocup.loc[
    oit_ocup["potential25"].isin(alta), "dummy_ia"
] = 1

oit_ocup.loc[
    oit_ocup["potential25"].isin(baixa), "dummy_ia"
] = 0





e agora padroniza e merge nelas

In [ ]:
pnad["cod_pnad"] = (
    pnad["V4010"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(4)
)

oit_ocup["isco08"] = (
    oit_ocup["ISCO_08"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(4)
)




In [ ]:
codigos_oit = set(oit_ocup["isco08"].dropna())
com_ocupacao = pnad["cod_pnad"].notna()
correspondencia = pnad["cod_pnad"].isin(codigos_oit)
print("Ocupações distintas na PNAD:",
      pnad["cod_pnad"].nunique())

print("Ocupações distintas na OIT:",
      oit_ocup["isco08"].nunique())

print("Trabalhadores com ocupação registrada:",
      com_ocupacao.sum())

print("Correspondência direta (%):",
      round(correspondencia[com_ocupacao].mean() * 100, 2))

# Códigos da PNAD sem correspondência direta
sem_match = pnad.loc[
    com_ocupacao & ~correspondencia,
    "cod_pnad"
]



In [ ]:
mapa_cod_isco = {
    "0411": "5412",
    "0412": "5412",
    "0511": "5411",
    "0512": "5411",
    "5168": "5169",
    "6225": "6220"}

pnad["isco08_ajustado"] = pnad["cod_pnad"].replace(
    mapa_cod_isco
)

colunas_oit = [
    "isco08",
    "Title",
    "score_ia",
    "potential25",
    "dummy_ia"
]

assert oit_ocup["isco08"].is_unique

base = pnad.merge(
    oit_ocup[colunas_oit],
    left_on="isco08_ajustado",
    right_on="isco08",                                  #aqui!
    how="left",
    validate="many_to_one",
    indicator="status_merge"
)

com_ocupacao = base["cod_pnad"].notna()
associados = base["status_merge"] == "both"

print("Observações da PNAD:", len(pnad))
print("Observações após merge:", len(base))

print(
    "Correspondência (%):",
    round(
        associados[com_ocupacao].mean() * 100, 2
    )
)

display(print(base.head()))

In [ ]:

# 1. Verificar se o merge preservou as observações
print("Observações na PNAD:", len(pnad))
print("Observações na base integrada:", len(base))

assert len(pnad) == len(base)

# 2. Verificar a cobertura do índice da OIT
com_ocupacao = base["cod_pnad"].notna()
com_score = base["score_ia"].notna()

print("\nTrabalhadores com ocupação:", com_ocupacao.sum())
print("Trabalhadores com escore de IA:", (com_ocupacao & com_score).sum())

# 3. Distribuição dos trabalhadores por exposição
print("\nDistribuição da exposição à IA:")
print(
    base.loc[com_score, "potential25"]
    .value_counts(dropna=False)
)

# 4. Distribuição da dummy
print("\nDistribuição da dummy de IA:")
print(
    base.loc[com_score, "dummy_ia"]
    .value_counts(dropna=False)
)


In [39]:

assert len(base) == len(pnad)

print("Observações:", len(base))
print("Variáveis:", len(base.columns))

print("\nIndicadores da OIT:")
print(base[["score_ia", "dummy_ia", "potential25"]].notna().sum())

print("\nStatus do cruzamento:")
print(base["status_merge"].value_counts())

Observações: 498494
Variáveis: 24

Indicadores da OIT:
score_ia       221388
dummy_ia       182121
potential25    221388
dtype: int64

Status do cruzamento:
status_merge
left_only     277106
both          221388
right_only         0
Name: count, dtype: int64


In [40]:
pasta_saida = raiz / "bases de dados" / "processadas"
pasta_saida.mkdir(parents=True, exist_ok=True)


arquivo_saida = pasta_saida / "BASE_BRUTA.xlsx"


base.to_excel(
    arquivo_saida,
    index=False,
    engine="openpyxl"
)